# PharmaTrace AI — Shipments-Driven Demand Prediction
## Strategic Engine 6: Clinical Demand Intelligence
---
**Data Source:** `shipments` sheet from `PharmaTrace_Master_Dataset_Production_Extended_Cleaned.xlsx`  
**Approach:** Product × Month aggregation → Rule-based pattern classification → XGBoost 1M/3M/6M forecasts  
**Why shipments (not monthly_demand):** The shipments sheet is the transactional source of truth — it has distributor_id, retailer_id, warehouse_id, and real quantities. The monthly_demand sheet was synthetically generated with pre-labeled categories.  

### Clinical Demand Pattern Rules (data-driven, no pre-labels):
| Priority | Pattern | Rule |
|---|---|---|
| 1 | CONTROLLED_SUBSTANCE_REGULATED | DEA schedule field (real FDA data) |
| 2 | SPECIALTY_ONCOLOGY_HIGH_VALUE | Unit price ≥$300 OR oncology pharm class + low volume |
| 3 | ACUTE_SEASONAL_WINTER_SURGE | Nov–Feb shipments ≥1.35× off-season mean (computed) |
| 4 | CHRONIC_MAINTENANCE_STEADY | All others (default) |

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
warnings.filterwarnings('ignore')

# Dark plot style
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#0f1117',
    'axes.edgecolor': '#334155', 'text.color': '#e2e8f0',
    'axes.labelcolor': '#94a3b8', 'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8', 'legend.framealpha': 0,
    'legend.labelcolor': '#e2e8f0', 'grid.color': '#1e293b',
    'font.family': 'DejaVu Sans'
})

PALETTE = ['#00d4ff','#10b981','#f59e0b','#7c3aed','#ef4444','#06b6d4','#84cc16','#fb923c']
PATTERN_COLORS = {
    'CHRONIC_MAINTENANCE_STEADY':       '#10b981',
    'ACUTE_SEASONAL_WINTER_SURGE':      '#f59e0b',
    'CONTROLLED_SUBSTANCE_REGULATED':   '#ef4444',
    'SPECIALTY_ONCOLOGY_HIGH_VALUE':    '#7c3aed',
}
print('Libraries loaded ✓')

In [ ]:
# ── DATA PATH ────────────────────────────────────────────────────────────────
DATA_PATH = os.path.join(
    os.path.dirname(os.getcwd()),
    'PharmaTrace AI - DATA', 'master_dataset',
    'PharmaTrace_Master_Dataset_Production_Extended_Cleaned.xlsx'
)

# Fallback: check data/ folder
if not os.path.exists(DATA_PATH):
    DATA_PATH = os.path.join(os.getcwd(), 'data',
        'PharmaTrace_Master_Dataset_Production_Extended_Cleaned.xlsx')

print(f'Data path: {DATA_PATH}')
print(f'Exists: {os.path.exists(DATA_PATH)}')

## Step 1: Load Sheets

In [ ]:
xl = pd.ExcelFile(DATA_PATH)
print('Sheets available:', xl.sheet_names)

df_ship = xl.parse('shipments')
df_fpb  = xl.parse('finished_product_batches')
df_prod = xl.parse('products')
df_dist = xl.parse('distributors')
df_ret  = xl.parse('retailers')
df_wh   = xl.parse('warehouses')

print(f'Shipments:   {len(df_ship):,} rows | {df_ship.shape[1]} cols')
print(f'Products:    {len(df_prod):,} rows')
print(f'Distributors:{len(df_dist):,} rows')
print(f'Retailers:   {len(df_ret):,} rows')
print(f'Warehouses:  {len(df_wh):,} rows')
df_ship.head(3)

## Step 2: Join & Enrich Shipments

In [ ]:
fpb  = df_fpb[['fp_batch_id','product_id']].drop_duplicates()
prod = df_prod[['product_id','generic_name','pharm_class','dosage_form',
                'dea_schedule','unit_price','shelf_life_months']].drop_duplicates('product_id')
dist = df_dist[['distributor_id','region']].drop_duplicates('distributor_id')
ret  = df_ret[['retailer_id','retailer_type']].drop_duplicates('retailer_id')
wh   = df_wh[['warehouse_id','warehouse_type','state','temp_controlled','capacity_units']].drop_duplicates('warehouse_id')

df = df_ship.copy()
df = df.merge(fpb,  on='fp_batch_id',         how='left')
df = df.merge(prod, on='product_id',           how='left')
df = df.merge(dist, on='distributor_id',       how='left')
df = df.merge(ret,  on='retailer_id',          how='left')
df = df.merge(wh.rename(columns={'warehouse_id':'origin_warehouse_id','state':'wh_state'}),
              on='origin_warehouse_id', how='left')

df['ship_date']    = pd.to_datetime(df['ship_date'], errors='coerce')
df['year']         = df['ship_date'].dt.year
df['month']        = df['ship_date'].dt.month
df['year_month']   = df['ship_date'].dt.to_period('M').astype(str)
df['is_delayed']   = (df['status'] == 'delayed').astype(int)
df['is_controlled']= df['dea_schedule'].notna().astype(int)
df['is_cold_chain']= df['temp_controlled'].fillna(False).astype(int)

print(f'Enriched: {len(df):,} rows | {df.product_id.nunique()} products | {df.origin_warehouse_id.nunique()} warehouses')
df[['shipment_id','product_id','generic_name','quantity','ship_date','distributor_id','dea_schedule','unit_price']].head(5)

## Step 3: Aggregate to Product × Month

In [ ]:
agg = df.groupby(['product_id','year_month','year','month'], as_index=False).agg(
    total_quantity          = ('quantity',            'sum'),
    num_shipments           = ('shipment_id',         'count'),
    num_unique_retailers    = ('retailer_id',         'nunique'),
    num_unique_warehouses   = ('origin_warehouse_id', 'nunique'),
    num_unique_distributors = ('distributor_id',      'nunique'),
    delay_rate              = ('is_delayed',          'mean'),
    dominant_carrier        = ('carrier',             lambda x: x.mode().iloc[0] if not x.mode().empty else 'UPS'),
    dominant_retailer_type  = ('retailer_type',       lambda x: x.mode().iloc[0] if not x.mode().empty else 'pharmacy'),
    dominant_wh_type        = ('warehouse_type',      lambda x: x.mode().iloc[0] if not x.mode().empty else 'regional'),
    dominant_region         = ('region',              lambda x: x.mode().iloc[0] if not x.mode().empty else 'South'),
    generic_name            = ('generic_name',        'first'),
    pharm_class             = ('pharm_class',         'first'),
    dosage_form             = ('dosage_form',         'first'),
    dea_schedule            = ('dea_schedule',        'first'),
    unit_price              = ('unit_price',          'first'),
    shelf_life_months       = ('shelf_life_months',   'first'),
    is_controlled           = ('is_controlled',       'first'),
    is_cold_chain           = ('is_cold_chain',       'first'),
)
agg['year_month_dt'] = pd.to_datetime(agg['year_month'] + '-01')
agg = agg.sort_values(['product_id','year_month_dt'])
print(f'Monthly grain: {len(agg):,} rows | {agg.year_month.nunique()} months | {agg.product_id.nunique()} products')
print(f'Date range: {agg.year_month.min()} → {agg.year_month.max()}')
agg.head(3)

## Step 4: Rule-Based Clinical Demand Pattern Classification

In [ ]:
SPECIALTY_KW = ['antineoplastic','oncol','immunosuppressant','biologic',
                'monoclonal','kinase inhibitor','checkpoint','pd-l1','pd-1']
SPEC_PRICE   = 300.0   # USD — high value threshold
SPEC_MAX_VOL = 300.0   # units/month — low volume signal
SEAS_IDX_THR = 1.35    # winter 35%+ above off-season
SEAS_CV_THR  = 0.20    # moderate variability

# Per-product statistics
pstats = agg.groupby('product_id')['total_quantity'].agg(prod_mean='mean', prod_std='std').reset_index()
pstats['cv'] = (pstats['prod_std'] / pstats['prod_mean'].replace(0, np.nan)).fillna(0)

# Seasonal index: winter (Nov-Feb) vs off-season mean
WINTER = {11, 12, 1, 2}
agg['_iw'] = agg['month'].isin(WINTER).astype(int)
seas = agg.groupby(['product_id','_iw'])['total_quantity'].mean().reset_index()
sp = seas.pivot(index='product_id', columns='_iw', values='total_quantity').reset_index()
sp.columns.name = None
sp = sp.rename(columns={0:'off_mean', 1:'win_mean'})
for c in ['off_mean','win_mean']:
    if c not in sp.columns: sp[c] = 1.0
sp['seasonal_index'] = (sp['win_mean'] / sp['off_mean'].replace(0, np.nan)).fillna(1.0)

agg = agg.merge(pstats[['product_id','prod_mean','cv']], on='product_id', how='left')
agg = agg.merge(sp[['product_id','seasonal_index']],     on='product_id', how='left')
agg['seasonal_index'] = agg['seasonal_index'].fillna(1.0)
agg['cv']             = agg['cv'].fillna(0)
agg.drop(columns=['_iw'], inplace=True)

def classify(row):
    # Rule 1: DEA controlled substance (REAL FDA data field — highest priority)
    if pd.notna(row.get('dea_schedule')) and str(row['dea_schedule']).strip():
        return 'CONTROLLED_SUBSTANCE_REGULATED'
    # Rule 2: Specialty/Oncology
    pc    = str(row.get('pharm_class','')).lower()
    price = float(row.get('unit_price', 0) or 0)
    vol   = float(row.get('prod_mean', 9999) or 9999)
    if (any(kw in pc for kw in SPECIALTY_KW) or price >= SPEC_PRICE) and vol <= SPEC_MAX_VOL:
        return 'SPECIALTY_ONCOLOGY_HIGH_VALUE'
    # Rule 3: Seasonal winter surge (computed from actual shipment time series)
    if row.get('seasonal_index', 1.0) >= SEAS_IDX_THR and row.get('cv', 0) >= SEAS_CV_THR:
        return 'ACUTE_SEASONAL_WINTER_SURGE'
    # Rule 4: Default chronic maintenance
    return 'CHRONIC_MAINTENANCE_STEADY'

agg['clinical_demand_pattern'] = agg.apply(classify, axis=1)
counts = agg['clinical_demand_pattern'].value_counts()
print('Pattern distribution:')
for p, c in counts.items():
    print(f'  {p:<44} {c:>6,} ({c/len(agg)*100:.1f}%)')

In [ ]:
# Visualize pattern distribution and seasonal profiles
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Pie chart
ax = axes[0]
pv = agg['clinical_demand_pattern'].value_counts()
clrs = [PATTERN_COLORS.get(p,'#64748b') for p in pv.index]
wedges, texts, autotexts = ax.pie(
    pv.values, labels=[p.replace('_','\n') for p in pv.index],
    colors=clrs, autopct='%1.1f%%', startangle=90,
    textprops={'fontsize': 8.5, 'color': '#e2e8f0'},
    wedgeprops={'edgecolor': '#0f1117', 'linewidth': 2}
)
for at in autotexts: at.set_color('#0f1117'); at.set_fontweight('bold')
ax.set_title('Clinical Demand Pattern Distribution\n(Rule-based from Shipments)', fontsize=11, color='#e2e8f0', fontweight='bold')

# Seasonal profiles
ax2 = axes[1]
sd = agg.groupby(['clinical_demand_pattern','month'])['total_quantity'].mean().reset_index()
ml = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
for i, (pat, grp) in enumerate(sd.groupby('clinical_demand_pattern')):
    grp = grp.sort_values('month')
    c = PATTERN_COLORS.get(pat, PALETTE[i%len(PALETTE)])
    ax2.plot(grp['month'], grp['total_quantity'], label=pat.replace('_',' '),
             lw=2.5, marker='o', markersize=5, color=c)
    ax2.fill_between(grp['month'], grp['total_quantity'], alpha=0.08, color=c)
ax2.axvspan(11,12.5,alpha=0.08,color='#f59e0b'); ax2.axvspan(0.5,2.5,alpha=0.08,color='#f59e0b')
ax2.set_xticks(range(1,13)); ax2.set_xticklabels(ml, fontsize=9)
ax2.set_title('Seasonal Demand Profiles (shaded=winter)', fontsize=11, color='#e2e8f0', fontweight='bold')
ax2.set_ylabel('Avg Monthly Units Shipped', fontsize=9)
ax2.legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.show()

## Step 5: Feature Engineering

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_fe = agg.sort_values(['product_id','year_month_dt']).copy()

# Lag features per product
for lag in [1, 2, 3, 6]:
    df_fe[f'lag_{lag}m'] = df_fe.groupby('product_id')['total_quantity'].shift(lag)

# Rolling features
for w in [3, 6]:
    df_fe[f'rolling_mean_{w}m'] = df_fe.groupby('product_id')['total_quantity'].transform(
        lambda x: x.shift(1).rolling(w, min_periods=1).mean())
df_fe['rolling_std_3m'] = df_fe.groupby('product_id')['total_quantity'].transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).std()).fillna(0)
df_fe['mom_growth'] = df_fe.groupby('product_id')['total_quantity'].pct_change().replace(
    [np.inf,-np.inf], 0).fillna(0)

# Time features
df_fe['is_winter']          = df_fe['month'].isin({11,12,1,2}).astype(int)
df_fe['is_q4']              = df_fe['month'].isin({10,11,12}).astype(int)
df_fe['month_sin']          = np.sin(2*np.pi*df_fe['month']/12)
df_fe['month_cos']          = np.cos(2*np.pi*df_fe['month']/12)
df_fe['months_since_start'] = ((df_fe['year_month_dt']-df_fe['year_month_dt'].min()).dt.days/30.44).astype(int)

# Cross-sectional share
df_fe['month_total']   = df_fe.groupby('year_month')['total_quantity'].transform('sum')
df_fe['product_share'] = df_fe['total_quantity'] / df_fe['month_total'].replace(0, np.nan)

# Categorical encoding
cat_cols = ['clinical_demand_pattern','dosage_form','dominant_wh_type',
            'dominant_carrier','dominant_retailer_type','dominant_region']
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df_fe[col+'_enc'] = le.fit_transform(df_fe[col].fillna('unknown').astype(str))
    encoders[col] = le

df_fe['unit_price']        = df_fe['unit_price'].fillna(df_fe['unit_price'].median())
df_fe['shelf_life_months'] = df_fe['shelf_life_months'].fillna(24)
df_fe['delay_rate']        = df_fe['delay_rate'].fillna(0)

print(f'Features ready: {len(df_fe):,} rows | {len(df_fe.columns)} columns')
print('Sample lag_1m non-null:', df_fe['lag_1m'].notna().sum())

## Step 6: Train XGBoost — 1M / 3M / 6M Horizons

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, r2_score

FEATURE_COLS = [
    'lag_1m','lag_2m','lag_3m','lag_6m',
    'rolling_mean_3m','rolling_std_3m','rolling_mean_6m','mom_growth',
    'month','year','month_sin','month_cos','is_winter','is_q4','months_since_start',
    'num_shipments','num_unique_retailers','num_unique_warehouses','num_unique_distributors',
    'delay_rate','dominant_carrier_enc','dominant_retailer_type_enc',
    'unit_price','shelf_life_months','is_controlled','is_cold_chain',
    'clinical_demand_pattern_enc','dosage_form_enc',
    'dominant_wh_type_enc','cv','seasonal_index','prod_mean',
    'dominant_region_enc','product_share',
]
avail = [f for f in FEATURE_COLS if f in df_fe.columns]

# Time-based split
all_months = sorted(df_fe['year_month'].unique())
n = len(all_months)
train_cut = all_months[int(n*0.60)]
val_cut   = all_months[int(n*0.80)]
print(f'Train <= {train_cut} | Val {train_cut} - {val_cut} | Test > {val_cut}')

def safe_mape(yt, yp):
    yt, yp = np.array(yt), np.array(yp)
    mask = yt > 0
    return mean_absolute_percentage_error(yt[mask], yp[mask])*100 if mask.sum()>0 else np.nan

results = {}
for horizon in [1, 3, 6]:
    print(f'\n--- {horizon}-month horizon ---')
    df2 = df_fe.copy()
    df2['tgt'] = df2.groupby('product_id')['total_quantity'].shift(-horizon)
    df2 = df2.dropna(subset=['tgt'])
    X = df2[avail].fillna(0)
    y = df2['tgt']
    tr = df2['year_month'] <= train_cut
    va = (df2['year_month'] > train_cut) & (df2['year_month'] <= val_cut)
    te = df2['year_month'] > val_cut
    X_tr, y_tr = X[tr], y[tr]
    X_va, y_va = X[va], y[va]
    X_te, y_te = X[te], y[te]
    print(f'  Train={len(X_tr):,} Val={len(X_va):,} Test={len(X_te):,}')
    if len(X_tr) < 50:
        print('  [SKIP] Insufficient data'); continue
    model = XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.05,
                         subsample=0.8, colsample_bytree=0.75, min_child_weight=5,
                         reg_alpha=0.1, reg_lambda=1.0, random_state=42, n_jobs=-1, verbosity=0)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    val_pred  = np.maximum(model.predict(X_va), 0)
    y_te_eff  = y_te if len(X_te) > 0 else y_va
    X_te_eff  = X_te if len(X_te) > 0 else X_va
    test_pred = np.maximum(model.predict(X_te_eff), 0)
    m = {
        'val_mape':  safe_mape(y_va, val_pred),
        'val_rmse':  float(np.sqrt(mean_squared_error(y_va, val_pred))),
        'val_r2':    float(r2_score(y_va, val_pred)),
        'test_mape': safe_mape(y_te_eff, test_pred),
        'test_rmse': float(np.sqrt(mean_squared_error(y_te_eff, test_pred))),
        'test_r2':   float(r2_score(y_te_eff, test_pred)),
    }
    print(f'  Val  MAPE={m["val_mape"]:.1f}%  RMSE={m["val_rmse"]:.0f}  R2={m["val_r2"]:.3f}')
    print(f'  Test MAPE={m["test_mape"]:.1f}%  RMSE={m["test_rmse"]:.0f}  R2={m["test_r2"]:.3f}')
    # Per-pattern MAPE
    df_te_plot = df2[te if len(X_te)>0 else va].copy()
    df_te_plot['pred'] = test_pred
    print('  Pattern MAPE:')
    for pat, grp in df_te_plot.groupby('clinical_demand_pattern'):
        pm = safe_mape(grp['tgt'].values, grp['pred'].values)
        print(f'    {pat:<44} {pm:.1f}%')
    fi = pd.DataFrame({'feature':avail,'importance':model.feature_importances_}).sort_values('importance',ascending=False)
    results[f'{horizon}m'] = {'model':model,'metrics':m,'feature_importance':fi}
print('\nTraining complete!')

## Step 7: Feature Importance & Model Evaluation Plots

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(7*len(results), 7))
if len(results) == 1: axes = [axes]
for ax, (hz, res) in zip(axes, results.items()):
    fi = res['feature_importance'].head(15)
    clrs = [PALETTE[i%len(PALETTE)] for i in range(len(fi))]
    ax.barh(fi['feature'][::-1], fi['importance'][::-1], color=clrs[::-1], alpha=0.85)
    ax.set_xlabel('Importance', fontsize=9)
    ax.set_title(f'Top 15 Features — {hz} Horizon', fontsize=11, color='#e2e8f0', fontweight='bold')
plt.suptitle('XGBoost Feature Importances by Forecast Horizon', fontsize=13, color='#e2e8f0', y=1.01)
plt.tight_layout(); plt.show()

# Metrics table
mdf = pd.DataFrame([{'Horizon':hz, **res['metrics']} for hz, res in results.items()])
print('\nModel Metrics Summary:')
display(mdf.round(3))

## Step 8: Forward-Looking Forecasts (1M / 3M / 6M)

In [ ]:
latest  = df_fe.sort_values('year_month_dt').groupby('product_id').last().reset_index()
last_dt = df_fe['year_month_dt'].max()
fc_rows = []
for h, res in results.items():
    hz = int(h[0])  # extract number from '1m'
    tgt_ym = (last_dt + pd.DateOffset(months=hz)).strftime('%Y-%m')
    X_fut  = latest[avail].fillna(0)
    preds  = np.maximum(res['model'].predict(X_fut), 0)
    tmp = latest[['product_id','generic_name','clinical_demand_pattern',
                   'dominant_wh_type','dominant_region','unit_price','pharm_class']].copy()
    tmp['forecast_month'] = tgt_ym
    tmp['horizon'] = h.upper()
    tmp['forecasted_qty'] = preds.round(0).astype(int)
    tmp['forecasted_value_usd'] = (preds * tmp['unit_price'].fillna(0)).round(2)
    fc_rows.append(tmp)

forecasts = pd.concat(fc_rows, ignore_index=True)
print(f'Forecasts: {len(forecasts):,} rows')
print('\nTop 10 highest-demand products (1M):')
display(forecasts[forecasts['horizon']=='1M'].nlargest(10,'forecasted_qty')[
    ['product_id','generic_name','clinical_demand_pattern','forecasted_qty','forecasted_value_usd']])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
for ax, hz in zip(axes, ['1M','3M','6M']):
    if hz not in forecasts['horizon'].values: continue
    fv = forecasts[forecasts['horizon']==hz].nlargest(15,'forecasted_qty')
    bc = [PATTERN_COLORS.get(p,'#64748b') for p in fv['clinical_demand_pattern']]
    ax.barh(fv['generic_name'].str[:28], fv['forecasted_qty'], color=bc, alpha=0.85, edgecolor='#1e293b')
    ax.set_xlabel('Forecasted Units', fontsize=9)
    ax.set_title(f'Top 15 Products — {hz} Forecast\n({fv["forecast_month"].iloc[0]})', fontsize=10, color='#e2e8f0', fontweight='bold')
    ax.invert_yaxis()

patches = [mpatches.Patch(color=c, label=p.replace('_',' ')) for p, c in PATTERN_COLORS.items()]
fig.legend(handles=patches, loc='lower center', ncol=4, fontsize=8.5, framealpha=0, labelcolor='#e2e8f0', bbox_to_anchor=(0.5,-0.05))
plt.suptitle('XGBoost Demand Forecasts — 1M / 3M / 6M Horizons', fontsize=13, color='#e2e8f0', y=1.01)
plt.tight_layout(); plt.show()

## Step 9: Warehouse & Distributor Demand Intelligence

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Warehouse demand
if 'origin_warehouse_id' in df.columns:
    wh_d = (agg.groupby(['origin_warehouse_id'] if 'origin_warehouse_id' in agg.columns
                        else ['dominant_wh_type'])['total_quantity'].sum()
                       .reset_index().sort_values('total_quantity', ascending=False))
    # Use raw shipments for warehouse breakdown
    wh_raw = df.groupby('origin_warehouse_id')['quantity'].sum().reset_index().sort_values('quantity',ascending=False)
    ax1 = axes[0]
    ax1.bar(wh_raw['origin_warehouse_id'], wh_raw['quantity']/1e3, color=PALETTE[:len(wh_raw)], alpha=0.85)
    ax1.set_ylabel('Total Units Shipped (K)', fontsize=9)
    ax1.set_title('Demand by Warehouse (Total Units from Shipments)', fontsize=11, color='#e2e8f0', fontweight='bold')
    ax1.tick_params(axis='x', rotation=30)

# Distributor demand (top 15)
dist_d = df.groupby('distributor_id').agg(Total=('quantity','sum'), Delay=('is_delayed','mean')).reset_index()
dist_d = dist_d.sort_values('Total', ascending=False).head(15)
ax2 = axes[1]
bc = ['#ef4444' if d>0.05 else '#10b981' for d in dist_d['Delay']]
ax2.barh(dist_d['distributor_id'], dist_d['Total']/1e3, color=bc, alpha=0.85)
ax2.set_xlabel('Total Units (K)', fontsize=9)
ax2.set_title('Top 15 Distributors by Volume (red=delay>5%)', fontsize=11, color='#e2e8f0', fontweight='bold')
ax2.invert_yaxis()
plt.tight_layout(); plt.show()

## Step 10: Procurement Action Plan

| Horizon | Action |
|---|---|
| 1M | Release purchase orders immediately — 60-day manufacturing lead time |
| 3M | Finalize supplier agreements and CMO batch schedules |
| 6M | Strategic API procurement and cold-chain capacity reservation |

**Per Pattern:**
- 🟢 **CHRONIC_MAINTENANCE_STEADY** → Auto min-max reorder, rolling 6-week interval
- ❄️ **ACUTE_SEASONAL_WINTER_SURGE** → Pre-season stock-build by October (75-day lead time)
- 🔴 **CONTROLLED_SUBSTANCE_REGULATED** → Pre-file DEA Form 222, 60 days in advance
- 💎 **SPECIALTY_ONCOLOGY_HIGH_VALUE** → Reserve cold-chain 3PL slots, 90-day booking

In [ ]:
print('=== PROCUREMENT ACTION PLAN ===')
SAFETY_STOCK_PCT = 0.18
for hz, res in results.items():
    h = int(hz[0])
    key = f'{h}M'
    fv = forecasts[forecasts['horizon']==key]
    if fv.empty: continue
    tot = int(fv['forecasted_qty'].sum())
    val = fv['forecasted_value_usd'].sum()
    buf = int(tot * SAFETY_STOCK_PCT)
    urg = '🚨 IMMEDIATE' if h==1 else ('⚠️  PLAN NOW' if h==3 else '🟢 SCHEDULE')
    print(f'\n{urg} — {hz.upper()} HORIZON')
    print(f'  Base forecast:   {tot:>10,} units')
    print(f'  +{int(SAFETY_STOCK_PCT*100)}% safety stock:  {buf:>10,} units')
    print(f'  Total order qty: {tot+buf:>10,} units')
    print(f'  Estimated value: ${val*1.18:>12,.0f} USD')
    print('  Pattern breakdown:')
    for pat, grp in fv.groupby('clinical_demand_pattern'):
        print(f'    {pat:<44} {int(grp["forecasted_qty"].sum()):>8,} units')